In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
from math import ceil
import re

In [ ]:
data_dir = Path("Q:") / "Neutron Data" / "1-Unconverted_Data" / "TB-61" / "UNFILTERED"
# [x for x in data_dir.iterdir()]

In [ ]:
csv_path_0 = data_dir / "SData_TB-61.CSV"
csv_path_1 = data_dir / "SData_TB-61_1.CSV"

In [ ]:
SEP = ";"

with open(csv_path_0) as csvfile:
    header_line = csvfile.readline().strip()
    data_line = csvfile.readline().strip()

psd_headers = [x for x in header_line.split(SEP) if x != "SAMPLES"]
data_count = len(data_line.split(SEP))
signal_headers = [str(n) for n in range(data_count - len(psd_headers))]
all_headers = psd_headers + signal_headers

In [ ]:
all_dtypes = [pl.String for _ in all_headers]

In [ ]:
lf0 = pl.scan_csv(csv_path_0, has_header=False, separator=SEP, skip_lines=1, schema_overrides=all_dtypes, new_columns=all_headers)
lf1 = pl.scan_csv(csv_path_1, has_header=False, separator=SEP, skip_lines=0, schema_overrides=all_dtypes, new_columns=all_headers)

In [ ]:
lfs = [lf0, lf1]
lf_concat = pl.concat(lfs)

In [ ]:
lf_psd = lf_concat.select(psd_headers)
lf_signals = lf_concat.select(signal_headers)
lf_sig_float = lf_signals.cast(pl.Float32)

In [ ]:
# lf_signals = lf_signals.cast(pl.Float32)
# lf_signals.cast(pl.Float32).head().collect()
print(lf_signals.explain())
print(lf_sig_float.explain())

In [ ]:
arr = np.array([[1, 2, 3, 4],[11, 12, 13, 14]])
schema = {k: pl.UInt16 for k in "abcd"}
df = pl.DataFrame(arr, schema)
df

In [ ]:
ph_schema = {"PULSE_HEIGHT": pl.Float32}


def get_pulse_heights(
    raw_signals_df: pl.DataFrame,
    baseline_idx_range: int = 40,
    baseline_offset: float = 0,
    max_adc: int = 16367,
    use_max_adc: bool = False
) -> pl.DataFrame:
    offset = int(baseline_offset * max_adc)
    signals_np = raw_signals_df.to_numpy()

    if use_max_adc:
        baselines = max_adc
    else:
        baselines = signals_np[
            :, :baseline_idx_range
        ].mean(axis=1).reshape(-1, 1)

    signals_np = -signals_np + baselines + offset
    pulse_heights = np.max(signals_np, axis=1)

    corrected_signals = pl.DataFrame(
        pulse_heights,
        ph_schema
    )
    return corrected_signals


lf_pulse_heights = lf_sig_float.map_batches(get_pulse_heights, projection_pushdown=False, schema=ph_schema).cast(pl.String)

In [ ]:
lf_psd_with_ph = pl.concat([lf_psd, lf_pulse_heights], how="horizontal")
# print(lf_psd_with_ph.collect_schema())
# print(lf_psd_with_ph.explain())
result, query_profile = lf_psd_with_ph.profile()
to_secs_exprs = [(pl.col(col_name).cast(pl.Float64) / 1e6) for col_name in ["start", "end"]]
query_profile = (
    query_profile
    .with_columns(*to_secs_exprs)
    .with_columns((pl.col("end") - pl.col("start")).alias("duration"))
)
print(query_profile)
result

In [ ]:
base_file_name = Path("testsave.parquet")
base_folder = Path() / "testdir"
pattern = r"(.+)_(\d+)\.parquet"
max_rows_per_part = 10000


def psd_file_path_fn(ctx: pl.BasePartitionContext) -> Path:
    file_stem = base_file_name.stem
    file_ext = base_file_name.suffix
    return f"psd_{file_stem}_{ctx.file_idx}{file_ext}"


def signals_file_path_fn(ctx: pl.BasePartitionContext) -> Path:
    file_stem = base_file_name.stem
    file_ext = base_file_name.suffix
    return f"signals_{file_stem}_{ctx.file_idx}{file_ext}"


sink_psd = lf_psd_with_ph.sink_parquet(
    pl.PartitionMaxSize(
        base_folder / "psd",
        file_path=psd_file_path_fn,
        max_size=max_rows_per_part,
    ),
    mkdir=True,
    lazy=True
)
sink_signals = lf_signals.sink_parquet(
    pl.PartitionMaxSize(
        base_folder / "signals",
        file_path=signals_file_path_fn,
        max_size=max_rows_per_part
    ),
    mkdir=True,
    lazy=True
)
pl.collect_all([sink_psd, sink_signals])
print("Done saving")

for subfolder in base_folder.iterdir():
    if not subfolder.is_dir():
        continue
    file_count = sum((1 for filepath in subfolder.iterdir() if filepath.suffix == ".parquet"))
    pad_length = len(str(file_count))
    for filepath in subfolder.iterdir():
        if filepath.suffix != ".parquet":
            continue
        # print(filepath.name)
        match = re.match(pattern, filepath.name)
        if match is None:
            continue
        filename_start, raw_idx = match.groups()
        file_idx = raw_idx.zfill(pad_length)
        new_filepath = filepath.parent / f"{filename_start}_{file_idx}.parquet"
        filepath.rename(new_filepath)
        # print(f"Renamed {filepath} to {new_filepath}")
print("Done renaming files")